In [107]:
import pandas as pd
from scipy import stats
import numpy as np
from pandas.api.types import CategoricalDtype
import re
from helpers import get_renamed_blocks, get_renamed_blocks3

In [108]:
pd.set_option('display.max_columns', None)

In [109]:
pd.set_option('display.max_columns', None)

In [110]:
pd.set_option('display.max_columns', None)

In [111]:
#STATES = ['in Betrieb', 'Gesetzlich an Stilllegung gehindert', 'Netzreserve',  'Sicherheitsbereitschaft', 'Sonderfall', 'vorläufig stillgelegt', 'stillgelegt', 'Kohlestromvermarktungsverbot']
#STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'KVBG','stillgelegt', 'vorläufig stillgelegt']
ACTIVE_STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'vorläufig stillgelegt']
ENERGIES = ["Kernenergie", "Braunkohle", "Steinkohle", "Erdgas", "Mineralölprodukte", "Abfall", "Biomasse", ""]

In [1024]:
bm = pd.read_csv("../basic/inspire_prtr_mapper.csv", engine="python")
plr = pd.read_excel("../data/Kraftwerksliste.xlsx", sheet_name=1, skiprows=10, decimal=',')
pl0 = pd.read_excel("../data/Kraftwerksliste.xlsx", skiprows=11, decimal=',')
prtr = pd.read_excel("../data/2023-12-08_PRTR-Deutschland_Freisetzungen.xlsx")

In [1025]:
bm['PRTR_Kennnummer'] = bm['PRTR_Kennnummer'].apply(lambda x: str(x).replace('/', '_'))
bm['InspireID_Betrieb'] = bm['InspireID_Betrieb'].apply(lambda x: str(x).replace('/', '_'))

In [1026]:
#bm.loc[bm.plantid.str.contains("_")]

In [1027]:
pl0_bak = pl0.copy()
plr_bak = plr.copy()

In [1028]:
list(pl0_bak)

['Unnamed: 0',
 'MaStR-Nr. der Stromerzeugungseinheit',
 'Anlagenbetreiber',
 'Anzeige-Name der Stromerzeugungseinheit',
 'PLZ der Einheit',
 'Ort der Einheit',
 'Straße der Einheit',
 'Hausnummer der Einheit',
 'Bundesland der Einheit',
 'Datum der erstmaligen Inbetriebnahme der Einheit',
 'Jahr der Inbetriebnahme der Einheit',
 'Kraftwerksstatus der Einheit',
 'Energieträger',
 'Hauptbrennstoff',
 'Speichertechnologie',
 'Auswertung Energieträger',
 'Wärmeauskopplung (KWK)\n(ja/nein)',
 'Ist die Stromerzeugungseinheit ein Bestandteil eines Grenzkraftwerkes?',
 'Bruttoleistung in MW',
 'Nettonennleistung (elektrische Wirkleistung) in MW',
 'Ist die Stromerzeugungseinheit ein Bestandteil eines Grenzkraftwerkes?: ja \nNettonennleistung der Einspeisung in ein deutsches Netz:',
 'Technologie der Stromerzeugung',
 'Volleinspeisung oder Teileinspeisung?',
 'Anschlussnetzbetreiber',
 'Spannungsebene']

In [1029]:
list(plr_bak)

['Unnamed: 0',
 'MaStR-Nr. der Stromerzeugungseinheit',
 'Anzeige-Name der Stromerzeugungseinheit',
 'PLZ der Einheit',
 'Ort der Einheit',
 'Straße der Einheit',
 'Hausnummer der Einheit',
 'Bundesland der Einheit',
 'Datum der erstmaligen Inbetriebnahme der Einheit (Datum/Jahr)',
 'Datum der endgültigen Stilllegung der Einheit (Datum/Jahr)',
 'Kraftwerksstatus der Einheit',
 'Energieträger',
 'Hauptbrennstoff',
 'Speichertechnologie',
 'Auswertung Energieträger',
 'Wärmeauskopplung (KWK)\n(ja/nein)',
 'Bruttoleistung in MW',
 'Nettonennleistung (elektrische Wirkleistung) in MW',
 'Technologie der Stromerzeugung',
 'Volleinspeisung oder Teileinspeisung?',
 'Anschlussnetzbetreiber',
 'Spannungsebene',
 'Unnamed: 22']

In [1030]:
cols = [0,12,14,17,20,21]
plt = get_renamed_blocks(pl0)
plq = plt.drop(plt.columns[cols],axis=1)
cols2 = [0,11,13,19,20,22]
plr0a = get_renamed_blocks3(plr)
plr0b = plr0a.drop(plr0a.columns[cols2],axis=1)

In [1031]:
#plr0a.sort_values('power')

In [1032]:
#list(plq)
#list(plr0b)

In [1033]:
list(plq)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'fullsupply',
 'TSO',
 'voltagelevel']

In [1034]:
list(plr0b)

['blockid',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'tech',
 'voltagelevel']

In [1035]:
plq0a = plq[~plq['blockid'].isin(["SEE9-Dummy-nicht_EEG", "SEE9-Dummy-EEG", "SEE9-Dummy-EE"])] # remove non-conventional facilities
plq0b = plq0a.sort_values(['power'], ascending=False)

In [1036]:
#plq0b

In [1037]:
plq0a.insert(10, "endop", np.nan)
plr0b.insert(1, "company", np.nan)

In [1038]:
plr0b.insert(9, "initialopYear", np.nan)

In [1039]:
list(plr0b)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'tech',
 'voltagelevel']

In [1040]:
list(plq0a)

['blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'initialop',
 'initialopYear',
 'endop',
 'state',
 'e2',
 'energysource',
 'chp',
 'grosspower',
 'power',
 'fullsupply',
 'TSO',
 'voltagelevel']

In [1041]:
plq0a.shape

(2122, 20)

In [1042]:
plr0b.shape

(299, 19)

In [1043]:
plq0a.shape

(2122, 20)

In [1044]:
plr0b.shape

(299, 19)

In [1045]:
pl_combined = pd.concat([plq0a, plr0b])
pl_combined['energysource'] = pl_combined['energysource'].apply(lambda x: str(x).replace('Wärme', 'Erdgas'))

In [1046]:
pl_combined

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023-04-05 00:00:00,2023.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001-06-01 00:00:00,2001.0,NaN,In Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025-05-08 00:00:00,2025.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,SEE954774732035,NaN,Gasmotor 11,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,NaN,NaN,Mittelspannung,Verbrennungsmotor
295,SEE956374067388,NaN,Gasmotor 12,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,NaN,NaN,Mittelspannung,Verbrennungsmotor
296,SEE978841169494,NaN,Gasmotor 13,66333.0,Völklingen,Saarbrücker Straße,135 - 137,Saarland,2004-06-01 00:00:00,NaN,2025-03-26 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,NaN,Grubengas,Ja,3.047,2.927,NaN,NaN,Mittelspannung,Verbrennungsmotor
297,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011-02-21 00:00:00,NaN,2025-04-30 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1047]:
pl_combined.loc[pl_combined.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [1048]:
pl_combined.shape

(2421, 21)

In [1049]:
pl0 = pl_combined.copy()

In [1050]:
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()

In [1051]:
pl1

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023-04-05 00:00:00,2023.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001-06-01 00:00:00,2001.0,NaN,In Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025-05-08 00:00:00,2025.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981-01-01 00:00:00,NaN,2025-01-01 00:00:00,endgültig stillgelegt 2025 (nach § 13b EnWG),"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
291,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014-07-31 00:00:00,NaN,2024-08-24 00:00:00,endgültig stillgelegt 2024 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
292,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958-01-01 00:00:00,NaN,2025-07-06 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
297,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011-02-21 00:00:00,NaN,2025-04-30 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1052]:
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]

In [1053]:
pl0.loc[pl0["blockid"] == "BNA0645"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech


In [1054]:
#pl_debug2.groupby("state").count()

In [1055]:
#pl_debug2.groupby("initialop").count()

In [1056]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_853045/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [1057]:
pl1.loc[pl1.blockid == "SEE975094128629"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
266,SEE975094128629,NaN,HKW Mitte Kessel 12,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,1971-07-15 00:00:00,NaN,2024-06-26 00:00:00,endgültig stillgelegt 2024 (nach § 13b EnWG),"Erdgas, Erdölgas",Erdgas,Ja,22.4,20.0,NaN,NaN,Hochspannung,Gegendruckmaschine mit Entnahme


In [1058]:
#pl_debug2['initialop'] = pl_debug.initialop.dt.year

In [1059]:
#pl_debug['initialop'] = pl_debug['initialop'].to_datetime()

In [1060]:
pl_debug2.groupby("state").count()

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
state,,,,,,,,,,,,,,,,,,,,
Endgültig Stillgelegt 2011 (ohne StA),5,0,5,5,5,0,0,5,5,0,5,0,5,5,0,5,0,0,5,0
Endgültig Stillgelegt 2012 (ohne StA),21,0,21,21,21,5,3,21,20,0,21,2,21,21,0,21,0,0,21,0
Endgültig Stillgelegt 2013 (mit StA),1,0,1,1,1,0,0,1,1,0,1,0,1,1,0,1,0,0,1,0
Endgültig Stillgelegt 2013 (ohne StA),10,0,8,10,10,4,4,10,10,0,10,3,10,10,0,10,0,0,10,0
Endgültig Stillgelegt 2014 (mit StA),8,0,8,8,8,3,2,8,8,0,8,1,8,7,0,8,0,0,8,0
Endgültig Stillgelegt 2014 (ohne StA),2,0,2,2,2,2,2,2,2,0,2,0,2,2,0,2,0,0,2,0
Endgültig Stillgelegt 2015 (mit StA),10,0,10,10,10,2,2,10,10,0,10,3,10,10,0,10,0,0,10,0
Endgültig Stillgelegt 2015 (ohne StA),1,0,1,1,1,1,1,1,1,0,1,0,1,1,0,1,0,0,1,0
Endgültig Stillgelegt 2016 (mit StA),11,0,11,11,11,7,7,11,11,0,11,3,11,11,0,11,0,0,11,0


In [1061]:
#pl_debug2.groupby("initialop").count()

In [1062]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_853045/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [1063]:
#pl0

In [1064]:
#bm

In [1065]:
#pl0.groupby("Energieträger").count()

In [1066]:
#list(plt)

In [1067]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

/tmp/ipykernel_853045/293089823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl1['state'] = pl1['state'].fillna("stillgelegt")


In [1068]:
pl1.loc[pl1.blockid == "BNA0221c"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
108,BNA0221c,NaN,Gasblock,40221.0,Düsseldorf,Auf der Lausward,75,Nordrhein-Westfalen,1977-03-28 00:00:00,NaN,2019,Endgültig Stillgelegt 2019 (mit StA),NaN,Erdgas,Ja,NaN,293.0,NaN,NaN,Hochspannung (HS),NaN


In [1069]:
plt.loc[plt.blockid == "BNA0711"]

,Unnamed: 0,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,state,e1,e2,Speichertechnologie,energysource,chp,border,grosspower,power,gerpower,tech,fullsupply,TSO,voltagelevel


In [1070]:
plt = pl1.dropna(subset=["blockid"])

In [1071]:
plt = plt.astype({"energysource": 'category'})

In [1072]:
plt.loc[plt.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [1073]:
pl2 = pl1.copy()

In [1074]:
def to_year(x):
    dt = pd.to_datetime(x)
    return dt.year

In [1075]:
def to_year2(x):
    if pd.isna(x):
        return x
    elif isinstance(x, float):
        return x
    elif isinstance(x, int):
        return x
    else:
        ts = pd.Timestamp(ts_input=x)
        #print(ts)
        return int(ts.year)

In [1076]:
#blocks.loc[blocks.Energieträger == "Kernenergie"]

In [1077]:
pl2

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023-04-05 00:00:00,2023.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001-06-01 00:00:00,2001.0,NaN,In Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009-05-01 00:00:00,2009.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025-05-08 00:00:00,2025.0,NaN,In Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981-01-01 00:00:00,NaN,2025-01-01 00:00:00,endgültig stillgelegt 2025 (nach § 13b EnWG),"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
291,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014-07-31 00:00:00,NaN,2024-08-24 00:00:00,endgültig stillgelegt 2024 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
292,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958-01-01 00:00:00,NaN,2025-07-06 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
297,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011-02-21 00:00:00,NaN,2025-04-30 00:00:00,endgültig stillgelegt 2025 (ohne § 13b EnWG od...,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1078]:
'''
pl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012
pl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002
pl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970
pl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951
pl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990
pl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953
pl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006
pl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012



pl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D

pl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld
'''

#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace('\n','')
#pl2['Nettoleistung'] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace(';','.')

"\npl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012\npl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002\npl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970\npl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951\npl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990\npl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953\npl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006\npl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012\n\n\n\npl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D\n\npl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld\n"

In [1079]:
#pl2[] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
pl2['initialop'] = pl2['initialop'].apply(to_year2)
#pl2['endop'] = pl2['endop'].apply(to_year2)
pl2['power'] = pl2['power'].apply(pd.to_numeric)

In [1080]:
pl2.loc[pl2.blockid == "BNA0711"]

,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
8,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,Endgültig Stillgelegt 2012 (ohne StA),NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [1081]:
#bm[pl2.duplicated(['BlockID'], keep=False)].sort_values("BlockID", ascending=False)

In [1082]:
#pl2[pl2.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [1083]:
#cols = [10,11,12,14,17,18,19]
#pl3 = pl2.drop(pl2.columns[cols],axis=1)

In [1084]:
#pl3.groupby('Unternehmen').max()

In [1085]:
#pl3

In [1086]:
def fix_company(company):
    #print(company)
    company_dict = {"RWE": "RWE AG", "Vattenfall": "Vattenfall GmbH", "Uniper": "Uniper SE", "EnBW": "EnBW AG", "Steag": "Steag GmbH", "Nordzucker": "Nordzucker AG", "Lausitz Energie": "LEAG"}
    for key, value in company_dict.items():
        if key in str(company):
            return value
    return company

In [1087]:
def extract_still(x):
    match = re.findall("[0-9]{4}",x)
    return match[0] if match else np.nan

In [1088]:
def fix_kwk(x):
    return "Nein" if x == "nein" else "Ja" if x == "ja" else x

In [1089]:
pl4 = pl2.copy()

In [1090]:
sq = 'Endgültig Stillgelegt 2013 (mit StA)'

In [1091]:
list(pl2.groupby("state").count().reset_index()['state'])

['Endgültig Stillgelegt 2011 (ohne StA)',
 'Endgültig Stillgelegt 2012 (ohne StA)',
 'Endgültig Stillgelegt 2013 (mit StA)',
 'Endgültig Stillgelegt 2013 (ohne StA)',
 'Endgültig Stillgelegt 2014 (mit StA)',
 'Endgültig Stillgelegt 2014 (ohne StA)',
 'Endgültig Stillgelegt 2015 (mit StA)',
 'Endgültig Stillgelegt 2015 (ohne StA)',
 'Endgültig Stillgelegt 2016 (mit StA)',
 'Endgültig Stillgelegt 2016 (ohne StA)',
 'Endgültig Stillgelegt 2017 (mit StA)',
 'Endgültig Stillgelegt 2017 (ohne StA)',
 'Endgültig Stillgelegt 2018 (mit StA)',
 'Endgültig Stillgelegt 2018 (ohne StA)',
 'Endgültig Stillgelegt 2019 (mit StA)',
 'Endgültig Stillgelegt 2019 (ohne StA)',
 'In Betrieb',
 'Kapazitätsreserve aufgrund von § 13e EnWG',
 'Netzreserve aufgrund von KVBG',
 'Netzreserve aufgrund § 13b EnWG',
 'besonderes netztechnisches Betriebsmittel',
 'endgültig stillgelegt 2017 (ohne § 13b EnWG oder KVBG)',
 'endgültig stillgelegt 2018 (ohne § 13b EnWG oder KVBG)',
 'endgültig stillgelegt 2020 (nach § 13b

In [1092]:
z = "endgültig stillgelegt 2022 (ohne § 13b EnWG oder KVBG)"

In [1093]:
def extract_state(state):
    if state == "In Betrieb":
        return "in Betrieb"
    elif "Endgültig Stillgelegt" in state:
        return "stillgelegt"
    elif "Vorläufig Stillgelegt" in state:
        return "vorläufig stillgelegt"
    elif "an Stilllegung gehindert" in state:
        return "Gesetzlich an Stilllegung gehindert"
    else:
        return state

In [1094]:
def extract_state2(state):
    
    states = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
    matches = ['In Betrieb', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve aufgrund § 13b EnWG', 'befristete Strommarktrückkehr', 'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'endgültig stillgelegt 20[0-9]{2} \(nach KVBG\)', r'[E|e]ndgültig [S|s]tillgelegt 20[0-9]{2} \([ohne]|[mit].']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [1095]:
def extract_state3(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [1096]:
extract_state3(sq)

'Endgültig Stillgelegt 2013 (mit StA)'

In [1097]:
#pl4

In [1098]:
pl4['company'] = pl4['company'].apply(lambda x: fix_company(x))
#pl4['endop'] = pl4['state'].apply(lambda x: extract_still(x))
pl4['state'] = pl4['state'].apply(lambda x: extract_state3(x))
pl4['chp'] = pl4['chp'].apply(lambda x: fix_kwk(x))

In [1099]:
list(pl4.groupby("state").count().reset_index()['state'])

['Endgültig Stillgelegt 2011 (ohne StA)',
 'Endgültig Stillgelegt 2012 (ohne StA)',
 'Endgültig Stillgelegt 2013 (mit StA)',
 'Endgültig Stillgelegt 2013 (ohne StA)',
 'Endgültig Stillgelegt 2014 (mit StA)',
 'Endgültig Stillgelegt 2014 (ohne StA)',
 'Endgültig Stillgelegt 2015 (mit StA)',
 'Endgültig Stillgelegt 2015 (ohne StA)',
 'Endgültig Stillgelegt 2016 (mit StA)',
 'Endgültig Stillgelegt 2016 (ohne StA)',
 'Endgültig Stillgelegt 2017 (mit StA)',
 'Endgültig Stillgelegt 2017 (ohne StA)',
 'Endgültig Stillgelegt 2018 (mit StA)',
 'Endgültig Stillgelegt 2018 (ohne StA)',
 'Endgültig Stillgelegt 2019 (mit StA)',
 'Endgültig Stillgelegt 2019 (ohne StA)',
 'KVBG',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'in Betrieb',
 'stillgelegt',
 'stillgelegt §13b EnWG',
 'vorläufig stillgelegt']

In [1100]:
atest = pl4.groupby("state").count()

In [1101]:
#atest

In [1102]:
newdf = pl4.copy()

In [1103]:
#newdf.dtypes

In [1104]:
newdf["state"] = newdf["state"].astype('category')
newdf["state"] = newdf["state"].cat.set_categories(STATES, ordered=True)

In [1105]:
newdf["chp"] = newdf["chp"].astype('category')
newdf["chp"] = newdf["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)

In [1106]:
pl3 = newdf

In [1107]:
atest = newdf.groupby(["power"]).count()
atest.sort_values("initialop", inplace=True, ascending=False)

In [1108]:
#print(atest)

In [1109]:
atest = pl3.groupby(["state"], observed=False).count()
# atest

In [1110]:
# pl3.rename(columns={ pl3.columns[10]: "Energieträger"}, inplace=True)

In [1111]:
#bm[bm.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [1112]:
#bm2 = bm.drop_duplicates()

In [1113]:
#pl3

In [1114]:
#bm

In [1115]:
pl4 = bm.merge(pl3, how="right", left_on="sseid", right_on="blockid")

In [1116]:
pl4

,InspireID_Betrieb,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
0,NaN,NaN,NaN,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,2023.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,22.535,21.505,Volleinspeisung,Regionetz GmbH (SNB911641710114),Mittelspannung,NaN
1,NaN,NaN,NaN,SEE973601397087,Hamburger Energiewerke GmbH,Tiefstack GuD DT40,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,NaN,Erdgas,Ja,28.000,27.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
2,NaN,NaN,NaN,SEE901401382521,Hamburger Energiewerke GmbH,Tiefstack HKW 16bar-Turbine,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2001.0,2001.0,NaN,in Betrieb,Wärme,Erdgas,Ja,5.000,4.900,Teileinspeisung (einschließlich Eigenverbrauch),Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
3,NaN,NaN,NaN,SEE999106955379,Hamburger Energiewerke GmbH,Tiefstack GuD GT42,22113,Hamburg,Andreas-Meyer-Straße,8,Hamburg,2009.0,2009.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,51.000,50.000,Volleinspeisung,Hamburger Energienetze GmbH (SNB968295079586),Hochspannung,NaN
4,NaN,NaN,NaN,SEE968161544286,Stadtwerke Duisburg AG,Mitte BHKW3 M1,47053,Duisburg,Bungertstr.,27,Nordrhein-Westfalen,2025.0,2025.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,4.500,4.400,Volleinspeisung,Netze Duisburg GmbH (SNB916123648602),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,NaN,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1117]:
#pl4.loc[pd.isnull(pl4['sseid'])]

In [1118]:
pl5 = pl4[pd.notnull(pl4['sseid'])]
#pl5 = pl4
pl5a = pl5[pd.notnull(pl5['power'])]
#pl6 =  pl5a.copy() #[pd.notnull(pl5a['plantid'])]
pl6 = pl5a.rename(columns={"InspireID_Betrieb": "plantid"})

In [1119]:
pl6

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
10,BWpf-450-80920445-00000000,06-08-80920445,SEE946736241334,SEE946736241334,Palm Power GmbH & Co. KG,HKW Aalen,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,2021.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,79.190,78.302,Volleinspeisung,Netze BW GmbH (SNB948311994307),Hochspannung,NaN
15,SD661-59,661-59,SEE902796244388,SEE902796244388,Sappi Alfeld,Turbine 4,31061,Alfeld,Mühlenmasch,1,Niedersachsen,1970.0,1970.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,14.000,11.000,Teileinspeisung (einschließlich Eigenverbrauch),Überlandwerk Leinetal GmbH (SNB910956210043),Mittelspannung,NaN
17,BWpf-450-1741292-00000000,06-08-1741292,SEE971282697380,SEE971282697380,EnBW AG,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,73776,Altbach,Industriestraße,11,Baden-Württemberg,1971.0,1971.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,50.000,45.000,Teileinspeisung (einschließlich Eigenverbrauch),Netze BW GmbH (SNB948311994307),Hochspannung,NaN
18,MV60004935,13-60-80624,SEE905257392765,SEE905257392765,Cosun Beet Company GmbH & Co. KG,Kesselhaus,17389,Anklam,Bluthsluster Straße,24,Mecklenburg-Vorpommern,1993.0,1993.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,16.000,15.000,Teileinspeisung (einschließlich Eigenverbrauch),E.DIS Netz GmbH (SNB941690671609),Mittelspannung,NaN
19,HE50000213,06-26200010633,SEE998387657606,SEE998387657606,VW Kraftwerk GmbH,VW Baunatal Dampfturbine,34225,Baunatal,Im Lehnhof,NaN,Hessen,2013.0,2013.0,NaN,in Betrieb,Dampf (zum Beispiel Prozesswärme),Erdgas,Ja,30.259,29.304,Teileinspeisung (einschließlich Eigenverbrauch),Avacon Netz GmbH (SNB990362338043),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,NaN,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1120]:
plnewtest = pl6.dropna(subset="plantid").sort_values("plantid")

In [1121]:
pl6

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
10,BWpf-450-80920445-00000000,06-08-80920445,SEE946736241334,SEE946736241334,Palm Power GmbH & Co. KG,HKW Aalen,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,2021.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,79.190,78.302,Volleinspeisung,Netze BW GmbH (SNB948311994307),Hochspannung,NaN
15,SD661-59,661-59,SEE902796244388,SEE902796244388,Sappi Alfeld,Turbine 4,31061,Alfeld,Mühlenmasch,1,Niedersachsen,1970.0,1970.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,14.000,11.000,Teileinspeisung (einschließlich Eigenverbrauch),Überlandwerk Leinetal GmbH (SNB910956210043),Mittelspannung,NaN
17,BWpf-450-1741292-00000000,06-08-1741292,SEE971282697380,SEE971282697380,EnBW AG,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,73776,Altbach,Industriestraße,11,Baden-Württemberg,1971.0,1971.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,50.000,45.000,Teileinspeisung (einschließlich Eigenverbrauch),Netze BW GmbH (SNB948311994307),Hochspannung,NaN
18,MV60004935,13-60-80624,SEE905257392765,SEE905257392765,Cosun Beet Company GmbH & Co. KG,Kesselhaus,17389,Anklam,Bluthsluster Straße,24,Mecklenburg-Vorpommern,1993.0,1993.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,16.000,15.000,Teileinspeisung (einschließlich Eigenverbrauch),E.DIS Netz GmbH (SNB941690671609),Mittelspannung,NaN
19,HE50000213,06-26200010633,SEE998387657606,SEE998387657606,VW Kraftwerk GmbH,VW Baunatal Dampfturbine,34225,Baunatal,Im Lehnhof,NaN,Hessen,2013.0,2013.0,NaN,in Betrieb,Dampf (zum Beispiel Prozesswärme),Erdgas,Ja,30.259,29.304,Teileinspeisung (einschließlich Eigenverbrauch),Avacon Netz GmbH (SNB990362338043),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025-01-01 00:00:00,NaN,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024-08-24 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025-07-06 00:00:00,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025-04-30 00:00:00,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1122]:
#plnewtest

In [1123]:
#pl6.dtypes

In [1124]:
pl6.loc[pl6.blockid == "BNA0711"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
1427,NW300-0326774,06-05-300-0326774,BNA0711,BNA0711,NaN,Niederaußem,50129.0,Bergheim,NaN,NaN,Nordrhein-Westfalen,1963.0,NaN,2012,NaN,NaN,Braunkohle,Nein,NaN,125.0,NaN,NaN,Höchstspannung (HöS),NaN


In [1125]:
pl6['initialop'] = pl6['initialop'].apply(pd.to_numeric)
pl6['endop'] = pl6['endop'].apply(lambda x: to_year2(x))

In [1126]:
#pl6.loc[pl6.blockid == "BNA0711"].dtypes

In [1127]:
pl6.loc[pl6.blockid == "SEE999977738880"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
1549,SD664-02,664-02,SEE999977738880,SEE999977738880,NaN,IKW Deuben,6682.0,Teuchern,Industriestraße,1,Sachsen-Anhalt,1936.0,NaN,2021.0,stillgelegt,Rohbraunkohlen,Braunkohle,Ja,78.0,67.0,NaN,NaN,Hochspannung,Gegendruckmaschine mit Entnahme


In [1128]:
blocks = pl6.copy()
stammdaten = pl6.copy()

In [1129]:
plants = blocks.copy()
#plants = plants.dropna(subset=["plantid", "initialop"]) #TODO: remove initialop from here

In [1130]:
plants

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
10,BWpf-450-80920445-00000000,06-08-80920445,SEE946736241334,SEE946736241334,Palm Power GmbH & Co. KG,HKW Aalen,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,2021.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,79.190,78.302,Volleinspeisung,Netze BW GmbH (SNB948311994307),Hochspannung,NaN
15,SD661-59,661-59,SEE902796244388,SEE902796244388,Sappi Alfeld,Turbine 4,31061,Alfeld,Mühlenmasch,1,Niedersachsen,1970.0,1970.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,14.000,11.000,Teileinspeisung (einschließlich Eigenverbrauch),Überlandwerk Leinetal GmbH (SNB910956210043),Mittelspannung,NaN
17,BWpf-450-1741292-00000000,06-08-1741292,SEE971282697380,SEE971282697380,EnBW AG,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,73776,Altbach,Industriestraße,11,Baden-Württemberg,1971.0,1971.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,50.000,45.000,Teileinspeisung (einschließlich Eigenverbrauch),Netze BW GmbH (SNB948311994307),Hochspannung,NaN
18,MV60004935,13-60-80624,SEE905257392765,SEE905257392765,Cosun Beet Company GmbH & Co. KG,Kesselhaus,17389,Anklam,Bluthsluster Straße,24,Mecklenburg-Vorpommern,1993.0,1993.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Ja,16.000,15.000,Teileinspeisung (einschließlich Eigenverbrauch),E.DIS Netz GmbH (SNB941690671609),Mittelspannung,NaN
19,HE50000213,06-26200010633,SEE998387657606,SEE998387657606,VW Kraftwerk GmbH,VW Baunatal Dampfturbine,34225,Baunatal,Im Lehnhof,NaN,Hessen,2013.0,2013.0,NaN,in Betrieb,Dampf (zum Beispiel Prozesswärme),Erdgas,Ja,30.259,29.304,Teileinspeisung (einschließlich Eigenverbrauch),Avacon Netz GmbH (SNB990362338043),Hochspannung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,NW900-0080266,06-05-900-0080266,SEE972461974179,SEE972461974179,NaN,Heizkraftwerk Hagen-Kabel H5,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,1981.0,NaN,2025.0,NaN,"Erdgas, Erdölgas",Erdgas,Ja,124.500,121.000,NaN,NaN,Hochspannung,Gasturbinen mit Abhitzekessel
1684,BYS00291,06-09-261-0007-0041,SEE909963484639,SEE909963484639,NaN,BMW Landshut KWK 3,84030.0,Landshut,Ohmstraße,2,Bayern,2014.0,NaN,2024.0,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.627,2.550,NaN,NaN,Mittelspannung,Verbrennungsmotor
1685,NW300-9046797,06-05-300-9046797,SEE956973616802,SEE956973616802,NaN,LEV KWG G12,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,1958.0,NaN,2025.0,stillgelegt,Steinkohlen,Steinkohle,Ja,18.000,17.235,NaN,NaN,Umspannebene Hochspannung/Mittelspannung,Gegendruckmaschine ohne Entnahme
1686,SD661-14,661-14,SEE932929596862,SEE932929596862,NaN,BHKW Hauffstraße,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,2011.0,NaN,2025.0,stillgelegt,"Erdgas, Erdölgas",Erdgas,Ja,2.016,1.968,NaN,NaN,Mittelspannung,Verbrennungsmotor


In [1131]:
plants["energysource"] = plants["energysource"].astype('category')
plants["energysource"] = plants["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants["state"] = plants["state"].astype('category')
#plants["state"] = plants["state"].cat.set_categories(STATES, ordered=True)
plants["chp"] = plants["chp"].astype('category')
plants["chp"] = plants["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants["federalstate"] = plants["federalstate"].astype('category')
plants["federalstate"] = plants["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [1132]:
pl1 = pl0.loc[pl0["energysource"].isin(["Kernenergie", "Erdgas", "Steinkohle", "Braunkohle", "Steinkohle"])]

In [1133]:
p_grp = plants.groupby("plantid")

In [1134]:
#plants.groupby("Energip_subgrper").max()

In [1135]:
plants.groupby("state", observed=False).count()

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
state,,,,,,,,,,,,,,,,,,,,,,,
in Betrieb,807,807,807,807,807,807,807,807,799,777,807,807,807,0,792,807,807,807,807,800,807,807,0
Kapazitätsreserve,12,12,12,12,12,12,12,12,12,11,12,12,12,0,12,12,12,12,12,12,12,12,0
Netzreserve,22,22,22,22,22,22,22,22,22,21,22,22,22,0,20,22,22,22,22,22,22,22,0
bnBm,14,14,14,14,14,14,14,14,14,14,14,14,14,0,13,14,14,14,14,14,14,14,0
KVBG,5,5,5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,0
stillgelegt,88,88,88,88,0,88,88,88,85,76,88,88,0,88,80,88,88,86,88,0,0,81,83
vorläufig stillgelegt,13,13,13,13,13,13,13,13,13,12,13,13,13,0,13,13,13,13,13,13,13,12,0


In [1136]:
ACTIVE_STATES

['in Betrieb',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'vorläufig stillgelegt']

In [1137]:
#plants.loc[plants['state'] == 'in Betrieb']

In [1138]:
#plants.loc[plants['state'].isin(ACTIVE_STATES)].sort_values('power', ascending=False)

In [1139]:
p_subgrp = plants.loc[plants['state'].isin(ACTIVE_STATES)].groupby('plantid') # TODO: add 

In [1140]:
plants_act = pd.DataFrame()
for plantid, group in p_subgrp:
    entry = {}
    entry["plantid"] = plantid
    entry["activepower"] = group["power"].sum()
    #plants_act = plants_act.append(entry, ignore_index=True)
    plants_act = pd.concat([plants_act, pd.DataFrame([entry])], ignore_index=True)

In [1141]:
plants_act

,plantid,activepower
0,06-02-B10117A007,239.000
1,BB16012651,1.957
2,BB16018798,176.000
3,BB23020389,4.700
4,BB23020490,333.500
...,...,...
281,ST18046,75.643
282,TH30013152,123.500
283,TH62013494,28.250
284,TH72012874,60.875


In [1144]:
plants_a = pd.DataFrame()
for plantid, group in p_grp:
    #if not plantid == "06-05-300-9046797":
    #    continue
    entry = {}
    #print(entry)
    entry["plantid"] = plantid
    # entry["BlockID"] = group
    try:
        entry["plantname"] = group["plantname"].value_counts().index[0]
    except IndexError:
        entry["plantname"] = np.nan
    entry["federalstate"] = group["federalstate"].value_counts().index[0]
    entry["energysource"] = group["energysource"].min()
    try:
        entry["chp"] = group["chp"].min()
    except TypeError:
        #print("aneror")
        #print(plantid)
        #print(group['KWK'].count())
        entry["chp"] = ""
    entry["latestexpanded"] = group["initialop"].max()
    entry["initialop"] = group["initialop"].min()
    entry["totalpower"] = group["power"].sum()
    entry["state"] = group["state"].min()
    entry["blockcount"] = group["blockid"].count()
    try:
        entry["company"] = group["company"].value_counts().index[0]
    except IndexError:
        ;
    #print(entry)
    #break
    plants_a = pd.concat([plants_a, pd.DataFrame([entry])], ignore_index=True)

In [1145]:
plants_a

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company
0,06-02-B10117A007,Tiefstack GuD GT41,Hamburg,Steinkohle,Ja,2009.0,1993.0,239.000,in Betrieb,2,Hamburger Energiewerke GmbH
1,06-05-100-0030723,HKW Elberfeld,Nordrhein-Westfalen,Steinkohle,Ja,1992.0,1992.0,85.000,NaN,1,NaN
2,06-05-100-0431554,KW Voerde,Nordrhein-Westfalen,Steinkohle,Nein,1985.0,1982.0,1390.000,NaN,2,NaN
3,06-05-100-0853075,KW West,Nordrhein-Westfalen,Steinkohle,Nein,1971.0,1971.0,640.000,NaN,2,NaN
4,16-30-31400103005,Heizkraftwerk Gera-Nord,Thüringen,Erdgas,Ja,1996.0,1996.0,74.000,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...
346,ST18046,Kraftwerk ZI DT 2,Sachsen-Anhalt,Erdgas,Ja,2017.0,1980.0,75.643,in Betrieb,8,K+S Minerals and Agriculture GmbH
347,TH30013152,DT,Thüringen,Erdgas,Ja,2022.0,1999.0,123.500,in Betrieb,5,SWE Energie GmbH
348,TH62013494,Kraftwerk UB DT1,Thüringen,Erdgas,Ja,2016.0,1964.0,28.250,in Betrieb,3,K+S Minerals and Agriculture GmbH
349,TH72012874,Gasmotor 1 Jena,Thüringen,Erdgas,Ja,2022.0,2022.0,60.875,in Betrieb,5,TEAG Thüringer Energie AG


In [1146]:
int_list = ['blockcount', 'initialop', 'latestexpanded']
for column in int_list:
    plants_a[column] = plants_a[column].astype(int)
#plant['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)

In [1147]:
#plants_a.dtypes

In [1148]:
plants_a["energysource"] = plants_a["energysource"].astype('category')
plants_a["energysource"] = plants_a["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants_a["state"] = plants_a["state"].astype('category')
plants_a["state"] = plants_a["state"].cat.set_categories(STATES, ordered=True)
plants_a["chp"] = plants_a["chp"].astype('category')
plants_a["chp"] = plants_a["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants_a["federalstate"] = plants_a["federalstate"].astype('category')
plants_a["federalstate"] = plants_a["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [1149]:
plants_a.loc[plants_a.plantid == "666-999"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [1150]:
plants_a.loc[plants_a.plantid == "06-00176010435"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [1151]:
plants_a.loc[plants_a.plantid == "06-05-900-0865327"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [1152]:
#plants_a = plants.drop_duplicates(subset="KraftwerkID")

In [1153]:
blocks.loc[blocks.sseid == "SEE930982693153"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech
832,NW300-9046030,06-05-300-9046030,SEE930982693153,SEE930982693153,Knapsack Power GmbH & Co. KG,Knapsack I - Gasturbine GT11,50354,Hürth,Industriestraße,300,Nordrhein-Westfalen,2007.0,2007.0,NaN,in Betrieb,"Erdgas, Erdölgas",Erdgas,Nein,308.0,308.0,Volleinspeisung,Amprion GmbH (SNB976890256486),Höchstspannung,NaN


In [1154]:
column_titles = ['plantid', 'plantname', 'federalstate','energysource', 'chp', 'latestexpanded', 'initialop', 'totalpower', 'state', 'blockcount', 'company']
plants_b = plants_a.reindex(columns=column_titles)

In [1155]:
blocks[blocks.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech


In [1156]:
plants_b.sort_values("initialop", ascending=False, inplace=True)

In [1157]:
plants_final = plants_b.loc[:]

In [1158]:
#plants_final.dtypes

In [1159]:
# plants_final

In [1160]:
# stammdaten

In [1161]:
#stammdaten

In [1162]:
# cols = [0,2,3,9,10,11,12]
stammdaten = pl6.copy()
drop_list = ['plantid', 'energysource', 'initialop', 'chp', 'plantname', 'power', 'state', 'endop', 'company', 'sseid', 'TSO', 'voltagelevel']
stammdaten_dafuq = stammdaten.drop(drop_list, axis=1)
stammdaten = stammdaten_dafuq.copy()

In [1163]:
stammdaten_dafuq.sort_values(['blockid'])

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech
1424,06-08-4380932,BNA0011,79774.0,Albbruck,NaN,NaN,Baden-Württemberg,NaN,Steinkohle,NaN,NaN,NaN
1519,03-07-07244141350,BNA0012d,31061.0,Alfeld,Mühlenmarsch,1,Niedersachsen,NaN,NaN,NaN,NaN,NaN
1446,06-26200010633,BNA0059a,34225.0,Baunatal,NaN,NaN,Hessen,NaN,NaN,NaN,NaN,NaN
1522,06-11-01-1105607,BNA0075,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
1505,06-11-01-1105607,BNA0076,12207.0,Berlin,Ostpreußendamm,61,Berlin,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
457,03-06-06060116320,SEE999585452499,30449,Hannover,Spinnereistraße,9,Niedersachsen,2011.0,"Erdgas, Erdölgas",82.35,Teileinspeisung (einschließlich Eigenverbrauch),NaN
825,07-01-126351,SEE999756197518,56727,Mayen,Polcher Straße,113,Rheinland-Pfalz,2015.0,"Erdgas, Erdölgas",18.50,Teileinspeisung (einschließlich Eigenverbrauch),NaN
253,06-05-100-0154540,SEE999839188105,40476,Düsseldorf,Rather Straße,51,Nordrhein-Westfalen,2012.0,"Erdgas, Erdölgas",4.30,Teileinspeisung (einschließlich Eigenverbrauch),NaN
1546,06-02-BERZ003800,SEE999848168525,21079.0,Hamburg,Moorburger Schanze,2,Hamburg,NaN,Steinkohlen,860.00,NaN,Kondensationsmaschine mit Entnahme


In [1164]:
stmp = pl6.copy()
stammdaten2 = stmp.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [1165]:
#stammdaten2

In [1166]:
DROP_ST = ['blockid', 'jahr', 'kennnummer','betriebsname','betriebsname_2','plz_x','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST2 = ['jahr', 'kennnummer','betriebsname','betriebsname_2','plz','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST3 = ['jahr', 'kennnummer', 'bundesland', 'flusseinzugsgebiet', "betriebsname", "betriebsname_2", 'taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']


In [1167]:
#DROP_ST = ['jahr']

In [1168]:
plants_d1 = plants_final.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [1169]:
#plants_d1.dtypes

In [1170]:
pld2 = plants_d1.drop_duplicates(['plantid'])

In [1171]:
pld3 = pld2.drop(DROP_ST3, axis=1)
plants_final1 = pld3

In [1172]:
#plants_act

In [1173]:
plants_final2 = plants_final1.merge(plants_act, on="plantid", how='left')

In [1174]:
plants_final3 = plants_final2.dropna(subset=["plantid"])

In [1175]:
plants_final3.groupby("energysource").count()

/tmp/ipykernel_853045/2887887728.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  plants_final3.groupby("energysource").count()


,plantid,plantname,federalstate,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
energysource,,,,,,,,,,,,,,,,,,,
Kernenergie,8,8,8,8,8,8,8,6,8,0,0,0,0,0,0,0,0,0,0
Braunkohle,28,28,28,28,28,28,28,27,28,23,0,0,0,0,0,0,0,0,23
Steinkohle,63,63,63,63,63,63,63,51,63,43,4,4,4,4,4,4,4,4,43
Erdgas,229,227,229,229,229,229,229,214,229,204,1,1,1,1,1,1,1,1,204
Mineralölprodukte,22,22,22,22,22,22,22,15,22,15,0,0,0,0,0,0,0,0,15
Abfall,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Biomasse,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [1176]:
plants_final2[pd.notnull(plants_final2['activepower'])].sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,Ja,2012,1979,2470.000,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.000
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.000,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.000
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.000
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.000
301,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2388.000,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1983.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,SN60018636,BHKW-G66-KWK-klein,Sachsen,Erdgas,Ja,43067,43067,3.736,in Betrieb,1,Volkswagen Sachsen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.736
65,SD661-80,0050 - STW - BHKW - KWK,Nordrhein-Westfalen,Erdgas,Ja,2017,2007,12.877,in Betrieb,5,Stadtwerke Kempen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.313
25,NW700-0104479,BHKW-G,Nordrhein-Westfalen,Erdgas,Ja,2016,2016,1.999,in Betrieb,1,Westag AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.999
24,NW100-0036701,BHKW BASF F17,Nordrhein-Westfalen,Erdgas,Ja,2016,2016,1.968,in Betrieb,1,BASF Personal Care and Nutrition GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.968


In [1177]:
plants_final2.sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
230,SN70015796,Boxberg Block N,Sachsen,Braunkohle,Ja,2012,1979,2470.0,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.0
322,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.0,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.0
275,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.0,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.0
217,BB45025564,Kraftwerk Jänschwalde Block A,Brandenburg,Braunkohle,Ja,1989,1981,3000.0,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.0
301,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2388.0,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1983.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,BWpf-450-1479296-00000000,Bestandsanlage_HKW_Aalen,Baden-Württemberg,Erdgas,Ja,1960,1960,15.0,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
336,NW100-0888309,HKW Venator Germany,Nordrhein-Westfalen,Erdgas,Ja,1960,1960,8.9,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
339,NW900-0271161,Shamrock,Nordrhein-Westfalen,Steinkohle,Ja,1957,1957,132.0,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,NW100-0081105,Frimmersdorf,Nordrhein-Westfalen,Braunkohle,Nein,1970,1957,2008.0,stillgelegt,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1178]:
plants_final2[pd.isnull(plants_final2['plantid'])]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower


In [1179]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech
10,06-08-80920445,SEE946736241334,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,"Erdgas, Erdölgas",79.190,Volleinspeisung,NaN
15,661-59,SEE902796244388,31061,Alfeld,Mühlenmasch,1,Niedersachsen,1970.0,"Erdgas, Erdölgas",14.000,Teileinspeisung (einschließlich Eigenverbrauch),NaN
17,06-08-1741292,SEE971282697380,73776,Altbach,Industriestraße,11,Baden-Württemberg,1971.0,"Erdgas, Erdölgas",50.000,Teileinspeisung (einschließlich Eigenverbrauch),NaN
18,13-60-80624,SEE905257392765,17389,Anklam,Bluthsluster Straße,24,Mecklenburg-Vorpommern,1993.0,"Erdgas, Erdölgas",16.000,Teileinspeisung (einschließlich Eigenverbrauch),NaN
19,06-26200010633,SEE998387657606,34225,Baunatal,Im Lehnhof,NaN,Hessen,2013.0,Dampf (zum Beispiel Prozesswärme),30.259,Teileinspeisung (einschließlich Eigenverbrauch),NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1683,06-05-900-0080266,SEE972461974179,58099.0,Hagen,Hohensyburgstraße,67,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",124.500,NaN,Gasturbinen mit Abhitzekessel
1684,06-09-261-0007-0041,SEE909963484639,84030.0,Landshut,Ohmstraße,2,Bayern,NaN,"Erdgas, Erdölgas",2.627,NaN,Verbrennungsmotor
1685,06-05-300-9046797,SEE956973616802,51373.0,Leverkusen,Kaiser-Wilhelm-Allee,80,Nordrhein-Westfalen,NaN,Steinkohlen,18.000,NaN,Gegendruckmaschine ohne Entnahme
1686,661-14,SEE932929596862,72762.0,Reutlingen,Hauffstraße,89,Baden-Württemberg,NaN,"Erdgas, Erdölgas",2.016,NaN,Verbrennungsmotor


In [1180]:
#plants_final2.sort_values("activepower", ascending=False)

In [1181]:
#stammdaten2.dtypes

In [1182]:
#stammdaten3.sort_values(by=['plantid'], ascending=True)

In [1183]:
#stammdaten3 = stammdaten2.drop(DROP_ST, axis=1)

In [1184]:
#stammdaten4 = stammdaten3.drop_duplicates(['bnaid'])

In [1185]:
#stammdaten4

In [1186]:
#list(stammdaten2)

In [1187]:
'''


"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,
blockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"
662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201

'''


'\n\n\n"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,\nblockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"\n662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201\n\n'

In [1188]:
#|664-02|IKW

In [1189]:
blocks2 = blocks[['blockid', 'plantid', 'plantname', 'federalstate', 'energysource', 'initialop', 'chp', 'power', 'state', 'endop', 'company']]

In [1190]:
blocks2['initialop'] = blocks2['initialop'].astype('Int32')
blocks2['endop'] = blocks2['endop'].astype('Int32')

/tmp/ipykernel_853045/1874890394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['initialop'] = blocks2['initialop'].astype('Int32')
/tmp/ipykernel_853045/1874890394.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['endop'] = blocks2['endop'].astype('Int32')


In [1191]:
blocks2.loc[blocks2.blockid == "06-08-2948214"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company


In [1192]:
blocks2.loc[blocks2.plantid == "BYS00041"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1158,SEE942366584926,BYS00041,SWM HKW Nord 1 T10,Bayern,Abfall,1991,Ja,18.0,in Betrieb,<NA>,SWM Services GmbH
1159,SEE952372080091,BYS00041,SWM HKW Nord 2 T20,Bayern,Erdgas,1991,Ja,333.0,in Betrieb,<NA>,SWM Services GmbH
1160,SEE998278854237,BYS00041,SWM HKW Nord 3 T30,Bayern,Abfall,1984,Ja,22.0,in Betrieb,<NA>,SWM Services GmbH


In [1193]:
blocks2['chp'] = blocks2['chp'].fillna('Nein')

/tmp/ipykernel_853045/2832682086.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2['chp'] = blocks2['chp'].fillna('Nein')


In [1194]:
list(blocks2)

['blockid',
 'plantid',
 'plantname',
 'federalstate',
 'energysource',
 'initialop',
 'chp',
 'power',
 'state',
 'endop',
 'company']

In [1195]:
stammdaten['plz'] = stammdaten['plz'].astype('Int32')

In [1196]:
stammdaten.loc[stammdaten.duplicated(subset=['blockid'])]

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,grosspower,fullsupply,tech


In [1197]:
stammdaten.drop_duplicates(subset="blockid", inplace=True)
blocks2.drop_duplicates(subset="blockid", inplace=True)
plants_final2.drop_duplicates(subset="plantid", inplace=True)

/tmp/ipykernel_853045/3735086467.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocks2.drop_duplicates(subset="blockid", inplace=True)


In [1198]:
stammdaten2.loc[stammdaten2.blockid == "SEE915851127786"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz_x,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,chp,grosspower,power,fullsupply,TSO,voltagelevel,tech,jahr,kennnummer,betriebsname,betriebsname_2,betreiber,eigentuemer,plz_y,ort,strasse,hausnr,bundesland,flusseinzugsgebiet,geo_lat_wgs84,geo_long_wgs84,taet_nr,taetigkeit,activity,haupttaetigkeit,branche,sector,nace_id,nace_wirtschaftszweig,nace_sector,stoffgruppe,substances_group,schadstoff,pollutant,umweltkompartiment,releases_to,jahresfracht_freisetzung,versehentliche_freisetzung,schadstoff_schwellenwert,einheit,unit,bestimmungsmethode,determination_method,schutzgrund_fracht,confidential_reason_release,schutzgrund_betrieb,confidential_reason_facility


In [1199]:
stammdaten2 = stammdaten[['blockid', 'plz', 'place', 'street', 'streetnum', 'federalstate']]

In [1200]:
#blocks2.loc['C', 'x'] = "BNA1949"

In [1201]:
stammdaten2.to_csv("stammdaten_nh_new.csv", index=False, header=False)
blocks2.to_csv("blocks_nh_2.csv", index=False, header=False)
blocks2.to_csv("blocks_new_nh.csv", index=False, header=False)
plants_final2.to_csv("plants_nh_2.csv", index=False, header=False)
stammdaten2.to_csv("stammdaten.csv", index=False)
blocks2.to_csv("blocks_2.csv", index=False)
plants_final2.to_csv("plants_2.csv", index=False)

In [1202]:
#sqlite3 plantwatch.db  "CREATE TABLE plants(plantid TEXT NOT NULL PRIMARY KEY, plantname TEXT, federalstate TEXT, energysource TEXT, chp TEXT, latestexpanded INT, initialop INT, totalpower REAL, state TEXT, blockcount INT,  company TEXT, plz TEXT, place TEXT, street TEXT, number TEXT, latitude REAL, longitude REAL, activepower REAL, energy_2015 INTEGER,  energy_2016 INTEGER,  energy_2017 INTEGER,  energy_2018 INTEGER,  energy_2019 INTEGER, energy_2020 INTEGER, energy_2021 INTEGER, co2_2007 INTEGER,  co2_2008 INTEGER,  co2_2009 INTEGER,  co2_2010 INTEGER,  co2_2011 INTEGER,  co2_2012 INTEGER,  co2_2013 INTEGER,  co2_2014 INTEGER,  co2_2015 INTEGER,  co2_2016 INTEGER,  co2_2017 INTEGER,  co2_2018 INTEGER, co2_2019 INTEGER, co2_2020 INTEGER, FOREIGN KEY (plantid) REFERENCES blocks(plantid) ON DELETE CASCADE);"


In [339]:
blocks2.dropna(subset=['plantid'])

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1,SEE946736241334,BWpf-450-80920445-00000000,HKW Aalen,Baden-Württemberg,Erdgas,2021,Nein,78.302,in Betrieb,<NA>,Palm Power GmbH & Co. KG
6,SEE902796244388,SD661-59,Turbine 4,Niedersachsen,Erdgas,1970,Nein,11.000,in Betrieb,<NA>,Sappi Alfeld
8,SEE944503176377,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau HKW 1,Baden-Württemberg,Steinkohle,1985,Nein,433.000,Netzreserve,<NA>,EnBW AG
9,SEE971282697380,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau Block 4 GT A (s...,Baden-Württemberg,Erdgas,1971,Nein,45.000,in Betrieb,<NA>,EnBW AG
10,SEE963975943188,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau GT B,Baden-Württemberg,Erdgas,1973,Nein,57.000,in Betrieb,<NA>,EnBW AG
...,...,...,...,...,...,...,...,...,...,...,...
1532,SEE996990805407,NI01010157480,HKW Mitte Kessel 1,Niedersachsen,Steinkohle,1985,Ja,44.500,stillgelegt,2024,NaN
1533,SEE975094128629,NI01010157480,HKW Mitte Kessel 12,Niedersachsen,Erdgas,1971,Ja,20.000,stillgelegt,2024,NaN
1535,SEE937157344278,BWpf-450-2142201-00000000,Kraftwerk Walheim Block 1,Baden-Württemberg,Steinkohle,1964,Nein,96.000,stillgelegt,2024,NaN
1536,SEE981220191160,BWpf-450-4124448-00000000,HKW Oberkirch,Baden-Württemberg,Steinkohle,1987,Ja,18.500,stillgelegt,2024,NaN


In [162]:
blocks2.shape

(1538, 11)

In [163]:
blocks2.dropna(subset=['plantid']).shape

(1096, 11)

In [164]:
#stammdaten.dropna(subset=['sseid', 'bnaid']).sort_values(by=['sseid'], ascending=False)

In [165]:
stammdaten

,PRTR_Kennnummer,blockid,plz,place,street,streetnum,federalstate,initialopYear,e2,gerpower,fullsupply,grosspower
0,NaN,SEE915851127786,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,2023.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
1,06-08-80920445,SEE946736241334,73432,Aalen,Palm Allee,1,Baden-Württemberg,2021.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
2,NaN,SEE930596800480,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
3,NaN,SEE929797382345,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1990.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
4,NaN,SEE988046628214,16356,Ahrensfelde,Lindenberger Straße,12,Brandenburg,1991.0,"Erdgas, Erdölgas",NaN,Volleinspeisung,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1533,03-01-01010157480,SEE975094128629,38114.0,Braunschweig,Reiherstraße,3,Niedersachsen,NaN,"Erdgas, Erdölgas",NaN,Volleinspeisung,22.400
1534,NaN,SEE926418118239,49479.0,Ibbenbüren,Groner Allee,76,Nordrhein-Westfalen,NaN,"Erdgas, Erdölgas",NaN,Teileinspeisung (einschließlich Eigenverbrauch),1.853
1535,06-08-2142201,SEE937157344278,74399.0,Walheim,Mühlstraße,1,Baden-Württemberg,NaN,Steinkohlen,NaN,Volleinspeisung,107.000
1536,06-08-4124448,SEE981220191160,77704.0,Oberkirch,Hauptstraße,2,Baden-Württemberg,NaN,Steinkohlen,NaN,Teileinspeisung (einschließlich Eigenverbrauch),20.000


In [166]:
blocks.loc[blocks["blockid"] == "BNA0645"]

,plantid,PRTR_Kennnummer,sseid,blockid,company,plantname,plz,place,street,streetnum,federalstate,initialop,initialopYear,endop,state,e2,energysource,power,gerpower,fullsupply,TSO,voltagelevel,chp,grosspower


In [167]:
#bm2.to_csv("bpm.csv", index=False)

In [168]:
#blocks

In [169]:
#blocks.groupby('Energieträger').count()

In [170]:
#pl4 = blocks.loc[blocks['Energieträger'] == "Mineralölprodukte"]

In [171]:
#pl4

In [172]:
#pd.set_option('display.max_rows', None)

In [173]:
#pl4.loc[pl4['Energieträger'] == "Mineralölprodukte"].sort_values(["Bundesland", "Unternehmen", "Kraftwerksname"])

In [174]:
#plants_final2

In [175]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988.0,Nein,1410.0,stillgelegt,2023.0,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986.0,Nein,1410.0,stillgelegt,2021.0,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985.0,Nein,1402.0,stillgelegt,2019.0,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984.0,Nein,1360.0,stillgelegt,2021.0,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988.0,Nein,1336.0,stillgelegt,2023.0,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989.0,Nein,1310.0,stillgelegt,2023.0,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984.0,Nein,1288.0,stillgelegt,2021.0,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984.0,Nein,1284.0,stillgelegt,2017.0,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982.0,Nein,1275.0,stillgelegt,2015.0,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012.0,NaN,1060.0,in Betrieb,NaN,RWE AG


In [293]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
1482,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.0,stillgelegt,2023,NaN
1423,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.0,stillgelegt,2021,NaN
1401,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.0,stillgelegt,2019,NaN
1426,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.0,stillgelegt,2021,NaN
1486,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.0,stillgelegt,2023,NaN
1491,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989,Nein,1310.0,stillgelegt,2023,NaN
1432,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984,Nein,1288.0,stillgelegt,2021,NaN
1374,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984,Nein,1284.0,stillgelegt,2017,NaN
1336,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982,Nein,1275.0,stillgelegt,2015,NaN
393,SEE993592183001,NW100-0248923,Neurath G,Nordrhein-Westfalen,Braunkohle,2012,NaN,1060.0,in Betrieb,<NA>,RWE AG
